# Bài 12
Đây là notebook chứa mã nguồn đầy đủ của bài 12.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai12(budget=100, w_gdp=0.35, w_equity=0.30, w_ai=0.25, risk_threshold=0.55):
    """Alias pipeline cho bài 12 (dùng trong test)."""
    return solve_bai12_dashboard(total_budget=budget * 1000, scenario='S5')

In [ ]:
def solve_bai12_dashboard(data_dir=None, total_budget=50000, scenario='S5'):
    """
    Dashboard tổng hợp AIDEOM-VN với 5 kịch bản chính sách.

    Args:
        data_dir: Đường dẫn data (không bắt buộc)
        total_budget: Ngân sách tổng (Tỷ VND hoặc nghìn tỷ tuỳ app)
        scenario: 'S1' đến 'S5'

    Returns:
        dict với 6 module kết quả
    """
    cfg = SCENARIOS.get(scenario, SCENARIOS['S5'])

    # Chuẩn hóa budget về đơn vị nghìn tỷ cho tính toán
    budget_k = total_budget / 1000 if total_budget > 500 else float(total_budget)

    # Chạy 6 module
    allocation   = _module2_lp_allocation(budget_k, cfg)
    gdp_forecast = _module1_cobb_douglas(budget_k, cfg)
    priority     = _module3_priority(cfg)
    labor_impact = _module4_labor(cfg)
    topsis       = _module5_topsis(cfg)
    risk         = _module6_risk(allocation, gdp_forecast, cfg)

    return {
        'scenario':     scenario,
        'scenario_name': cfg['desc'],
        'description':  cfg['desc'],
        'budget':       total_budget,

        # Module 1 - GDP forecast
        'gdp_forecast': gdp_forecast,

        # Module 2 - Budget allocation
        'allocation': allocation,

        # Module 3 - Sector priority
        'priority': priority,

        # Module 4 - Labor impact
        'labor_impact': labor_impact,

        # Module 5 - TOPSIS region ranking
        'topsis': topsis,

        # Module 6 - Risk assessment
        'risk': risk,

        # Radar data tổng hợp
        'radar': {
            'dimensions': ['GDP', 'Hạ tầng', 'AI', 'Công bằng', 'Nhân lực', 'Rủi ro thấp'],
            'values': [
                min(95, max(20, gdp_forecast['gdp'][-1] / 20 * 100)),
                min(95, max(20, allocation['I'] / budget_k * 100 * 2)),
                min(95, max(20, allocation['AI'] / budget_k * 100 * 3)),
                min(95, max(20, allocation['H'] / budget_k * 100 * 3)),
                min(95, max(20, allocation['D'] / budget_k * 100 * 2.5)),
                min(95, max(20, (1 - risk['risk_score']) * 100)),
            ]
        },

        # Scenario comparison
        'all_scenarios': {
            sc: SCENARIOS[sc]['desc'] for sc in SCENARIOS
        },
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai12_dashboard()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())